# Label every code-only false alarm (Colab)

This is the Colab run behind slide 9 and `results/final/false_alarm_auto_*`.
It reads the saved model answers from Google Drive, finds every **code-only false alarm** (a "vulnerable" answer on
the fixed version) for Claude Haiku 4.5, Gemini 3 Flash and GPT-5 Mini, and labels each one by the code its
explanation quotes:

* **quotes the fix**: quotes of lines the patch added or changed are at least as many as quotes of unchanged lines;
* **unchanged code**: the quotes mostly hit lines the patch did not touch;
* **no code cited**: nothing quoted matches the shown code.

No API calls and no keys: it only reads `MyDrive/14795-interproc-vuln-pairs/multi_eval/<model>/responses.jsonl`.
The rule was fixed before comparing it with the 24 hand labels; two variants are reported as sensitivity.
Run time is about a minute.

In [ ]:
# 1. Mount Drive (Colab asks you to approve access to your Google Drive).
from google.colab import drive
drive.mount('/content/drive')
RESPONSES = '/content/drive/MyDrive/14795-interproc-vuln-pairs/multi_eval'
OUT_DRIVE = '/content/drive/MyDrive/14795-interproc-vuln-pairs/results_final'

In [ ]:
# 2. Get this repository and the authors' data (pinned commit).
import os, subprocess, sys
REPO_URL = 'https://github.com/jlgao-cmu-F25/interproc-vuln-pairs.git'
REPO_REF = 'main'   # the branch or commit that contains repro/label_false_alarms.py
os.chdir('/content')
if not os.path.isdir('interproc-vuln-pairs'):
    subprocess.run(['git', 'clone', '-q', REPO_URL], check=True)
os.chdir('/content/interproc-vuln-pairs')
subprocess.run(['git', 'fetch', '-q', 'origin', REPO_REF], check=True)
subprocess.run(['git', 'checkout', '-q', 'FETCH_HEAD'], check=True)
subprocess.run([sys.executable, '-m', 'repro.setup_data'], check=True)  # clones sw-vuln at the pinned commit
print(subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'], text=True))

In [ ]:
# 3. Label all code-only false alarms and compare with the 24 hand labels.
r = subprocess.run([sys.executable, '-m', 'repro.label_false_alarms', '--responses', RESPONSES,
                    '--out', 'results/final'], capture_output=True, text=True)
print(r.stdout); print(r.stderr[-2000:])

In [ ]:
# 4. Results: main rule plus two sensitivity variants, and agreement with the hand labels.
import json, pandas as pd
summary = pd.read_csv('results/final/false_alarm_auto_summary.csv')
display(summary)
agreement = json.load(open('results/final/false_alarm_auto_agreement.json'))
for rule, a in agreement.items():
    print(f"{rule}: agrees with {a['agree']} of {a['n']} hand labels")
    for d in a['disagreements']:
        print('   ', d)

In [ ]:
# 5. Keep a copy on Drive.
import shutil
os.makedirs(OUT_DRIVE, exist_ok=True)
for name in ['false_alarm_auto_labels.csv', 'false_alarm_auto_summary.csv', 'false_alarm_auto_agreement.json']:
    shutil.copy(os.path.join('results/final', name), OUT_DRIVE)
print('copied to', OUT_DRIVE)